# Process the PubMed corpus into serialized graphs

This notebook reads the canonical `corpus_articles.parquet`, optionally filters rows by pathogen and publication year, embeds each `node_context.summary`, and saves one NetworkX graph per corpus row under `GRAPH_FOLDER`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib.util
import subprocess
import sys
import warnings
warnings.filterwarnings('ignore', message='IProgress not found.*', module='tqdm.auto')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if importlib.util.find_spec('sentence_transformers') is None:
    print('Installing the optional embedding dependency into the active notebook kernel...')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-e',
        f'{PROJECT_ROOT}[embeddings]',
    ])
from sentence_transformers import SentenceTransformer
from graphicalizer import (
    ExtractionDensityConfig,
    Graphicalizer,
    GraphicalizerConfig,
    NetworkXGraphStore,
    NodeContextConfig,
    NodeEmbeddingConfig,
    load_corpus_articles,
    load_ontology,
    process_corpus_articles,
)

In [ ]:
ASSETS_ROOT = PROJECT_ROOT / 'assets'
CORPUS_PATH = PROJECT_ROOT / 'outputs' / 'pubmed_screening' / 'corpus_articles.parquet'
CORPUS_PATHOGENS = None  # e.g. ['Nipah virus']
CORPUS_START_YEAR = None
CORPUS_END_YEAR = None
GRAPH_FOLDER = PROJECT_ROOT / 'outputs' / 'graphs'
GRAPH_ID_PREFIX = 'pubmed'
CONTINUE_ON_ERROR = True

ONTOLOGY_PATH = (
    ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology_assembled.yaml'
    if (ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology_assembled.yaml').exists()
    else ASSETS_ROOT / 'ontologies' / 'entity_ontology_microbiology.yaml'
)
LLM_PROVIDER = 'openai'  # choose 'openai' or 'ollama'
LLM_MODEL = 'gpt-4o-mini' if LLM_PROVIDER == 'openai' else 'gemma4:12b-mlx'
LLM_OPTIONS = (
    {'max_output_tokens': 16384}
    if LLM_PROVIDER == 'openai'
    else {'num_ctx': 32768, 'num_predict': -1}
)
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)

ontology = load_ontology(ONTOLOGY_PATH)
density = ExtractionDensityConfig(
    entities_per_word=0.05,
    relations_per_entity=1.5,
    minimum_entity_fraction=0.75,
    density_retries=2,
)
config = GraphicalizerConfig(
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    extraction_density=density,
    node_context=NodeContextConfig(),
    prompt_template_path=ASSETS_ROOT / 'prompts' / 'graphicalizer_prompt_template.yaml',
    prompt_snapshot_path=PROJECT_ROOT / 'outputs' / 'prompts' / 'ontology-aware-graphicalizer-0.4.0.yaml',
    context_policy='all_nodes',
)
graphicalizer = Graphicalizer.from_provider(
    ontology,
    config,
    options=LLM_OPTIONS,
    embedding_model=EMBEDDING_MODEL,
    embedding_config=NodeEmbeddingConfig(model_id=EMBEDDING_MODEL_NAME),
)
graph_store = NetworkXGraphStore(GRAPH_FOLDER)

In [ ]:
corpus = load_corpus_articles(
    CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
print('Filtered corpus rows:', len(corpus))
batch = process_corpus_articles(
    corpus,
    graphicalizer,
    graph_store,
    corpus_path=CORPUS_PATH,
    graph_id_prefix=GRAPH_ID_PREFIX,
    continue_on_error=CONTINUE_ON_ERROR,
    verbose=True,
)

print('Discovered:', batch.discovered)
print('Processed:', batch.processed)
print('Failed:', batch.failed)
print('Manifest:', batch.manifest_path)
if batch.failures:
    print('Failures:')
    for failure in batch.failures:
        print(failure)